# Download Data

In [ ]:
import pandas as pd

df = pd.read_parquet('../data/Player-Games_Injuries_Travel_Bio.parquet')
df = df.drop(columns=['days_to_next_injury', 'Name', 'coachId'])
df.head()

,personId,gameId,gameDateTimeEst_player,playerteamName,gameType_player,win_player,home_player,numMinutes_player,points_player,assists_player,...,direction_ew,tz_shift_hrs,rest_days,flight_minutes,height,weight,birthdate,experience,bmi,age
0,255.0,21000521,2011-01-05 22:30:00,Suns,Regular Season,0.0,1.0,34.0,10.0,2.0,...,East,1.0,2.0,69.3,80.0,225.0,1972-10-05,18,24.714844,38.250513
1,255.0,21000533,2011-01-07 22:30:00,Suns,Regular Season,0.0,1.0,29.0,10.0,2.0,...,N/A,0.0,1.0,0.0,80.0,225.0,1972-10-05,18,24.714844,38.255989
2,255.0,21000584,2011-01-14 22:30:00,Suns,Regular Season,1.0,1.0,36.0,21.0,3.0,...,N/A,0.0,1.0,0.0,80.0,225.0,1972-10-05,18,24.714844,38.275154
3,255.0,21000598,2011-01-17 13:00:00,Suns,Regular Season,1.0,0.0,37.0,25.0,2.0,...,East,2.0,2.0,234.0,80.0,225.0,1972-10-05,18,24.714844,38.283368
4,255.0,21000613,2011-01-19 19:00:00,Suns,Regular Season,1.0,0.0,40.0,27.0,2.0,...,West,0.0,1.0,44.1,80.0,225.0,1972-10-05,18,24.714844,38.288843


In [ ]:
%pip install catboost

In [ ]:
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score
import numpy as np
from sklearn.impute import SimpleImputer
from catboost import CatBoostClassifier
import matplotlib.pyplot as plt
import shap

def sensitivity_analysis_injury_prediction(df, target, fig_title):
    seasons = sorted(df['season_x'].unique())

    test_season = seasons[-1]
    val_season = seasons[-2]
    train_seasons = seasons[:-2]

    test_df = df[df['season_x'] == test_season]
    val_df = df[df['season_x'] == val_season]
    train_df = df[df['season_x'].isin(train_seasons)]

    # From 14-day prediction
    features = ['rolling_3g_three_pointers_made', 'rolling_7g_blocks', 'rolling_3g_three_pointers_attempted',
                'rolling_7g_steals', 'rolling_3g_steals', 'games_last_14d', 'rolling_7g_points',
                'rolling_3g_points_per36', 'rolling_3g_three_point_percent', 'rolling_3g_assists_per36',
                'rolling_3g_USG', 'rolling_7g_three_point_percent', 'rolling_7g_USG', 'positionless_index',
                'rolling_7g_3p_relative_percent', 'rolling_7g_assists', 'rolling_7g_assists_per36',
                'rolling_3g_field_goals_attempted', 'rolling_3g_assists', 'rolling_3g_points', 'rolling_3g_rebounds',
                'rolling_7g_points_per36', 'rolling_3g_minutes', 'winPercent_team', 'rolling_3g_field_goals_made',
                'rolling_7g_three_pointers_made', 'rolling_7g_field_goals_made', 'rolling_7g_three_pointers_attempted',
                'rolling_3g_3p_relative_percent', 'rolling_3g_blocks', 'rolling_7g_field_goals_attempted',
                'rolling_3g_turnovers', 'rolling_7g_ft_relative_percent']

    X_sel = df[features + ['season_x']].copy()
    y_sel = df[target].values

    train_mask = X_sel['season_x'].isin(train_seasons)
    val_mask = X_sel['season_x'] == val_season
    test_mask = X_sel['season_x'] == test_season

    X_train = X_sel.loc[train_mask].drop(columns=['season_x'])
    X_val = X_sel.loc[val_mask].drop(columns=['season_x'])
    X_test = X_sel.loc[test_mask].drop(columns=['season_x'])

    y_train = y_sel[train_mask]
    y_val = y_sel[val_mask]
    y_test = y_sel[test_mask]

    X_full_train = pd.concat([X_train, X_val], axis=0)
    y_full_train = np.concatenate([y_train, y_val])

    imputer = SimpleImputer(strategy='median')
    imputer.set_output(transform="pandas")
    X_full_train = imputer.fit_transform(X_full_train)
    X_test = imputer.transform(X_test)

    model = CatBoostClassifier(
            iterations=500,
            learning_rate=0.05,
            depth=6,
            loss_function="Logloss",
            eval_metric="AUC",
            verbose=0,
            random_seed=42
        )

    model.fit(X_full_train, y_full_train)
    preds = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, preds)
    print(f'AUC: {auc}')
    print(f'Baseline: {y_test.mean()}')
    ap = average_precision_score(y_test, preds)
    print(f'Average Precision Score: {ap}')

    probs = pd.DataFrame({
        "y": y_test,
        "p": preds
    }).sort_values("p", ascending=False)

    for pct in [1, 5, 10, 20]:
        n = int(len(probs) * pct / 100)
        precision = probs.head(n)["y"].mean()
        print(f"Top {pct}%: {precision:.3f}")

    precision, recall, _ = precision_recall_curve(y_test, preds)
    plt.plot(recall, precision, marker='.')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.show()

    explainer = shap.TreeExplainer(model)

    feature_renaming = {
        "rolling_7g_three_pointers_attempted": '3PA (Rolling: 7 Games)',
        'rolling_7g_USG': 'Usage Rate (Rolling: 7 Games)',
        'positionless_index': 'Positionless Index',
        'rolling_7g_points_per36': 'Points per 36 Minutes (Rolling: 7 Games)',
        'rolling_7g_rebounds': 'Rebounds (Rolling: 7 Games)',
        'winPercent_team': 'Team Win Percent',
        'rolling_7g_minutes': 'Minutes (Rolling: 7 Games)',
        'rolling_7g_blocks': 'Blocks (Rolling: 7 Games)',
        'games_last_14d': 'Games Played over the Last 14 Days',
        'rolling_7g_assists': 'Assists (Rolling: 7 Games)',
        'rolling_3g_rebounds': 'Rebounds (Rolling: 3 Games)',
        'rolling_3g_minutes': 'Minutes (Rolling: 3 Games)',
        'rolling_7g_assists_per36': 'Assists per 36 Minutes (Rolling: 7 Games)'
    }

    shap_values = explainer(X_test)

    shap_values.feature_names = [feature_renaming[c] if c in feature_renaming else c for c in X_test.columns]

    shap.plots.beeswarm(shap_values)

    fig = plt.gcf()

    fig.savefig(
        f"../output/Figures/{fig_title}.svg",
        format="svg",
        bbox_inches="tight"
    )
    
    plt.close()


In [ ]:
sensitivity_analysis_injury_prediction(df, 'injury_within_30d', '30-Day CatBoost SHAP Summary Plot')
sensitivity_analysis_injury_prediction(df, 'injury_within_7d', '7-Day CatBoost SHAP Summary Plot')
